In [ ]:
!pip install -q transformers datasets librosa evaluate jiwer gradio bitsandbytes accelerate
!pip install -q git+https://github.com/huggingface/peft.git@main
!pip install torchcodec
!pip install -U datasets[audio]


In [ ]:
from datasets import load_dataset, DatasetDict, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration, AutoModelForCausalLM, BitsAndBytesConfig, Seq2SeqTrainingArguments, Seq2SeqTrainer, TrainerCallback, TrainingArguments, TrainerState, TrainerControl
from transformers.trainer_utils import PREFIX_CHECKPOINT_DIR
from dataclasses import dataclass
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model, PeftModel, LoraModel, PeftConfig
from typing import Any, Dict, List, Union
import torch
import evaluate
import os
import gc
import numpy as np
from tqdm import tqdm
from torch.utils.data import DataLoader
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
DATASET_SUARA_GABUNGAN = 'grandhigh/SuaraGabungan-ID'
DATASET_INDO_CORPUS = '../../data/indo_corpus/id/'
OUTPUT_PATH = '../../result/saved_model/'
LANG = 'id'
TASK = 'transcribe'
MODEL_NAME = 'openai/whisper-base'

In [ ]:
data = load_dataset(DATASET_SUARA_GABUNGAN, split='train')
train_temp = data.train_test_split(test_size=0.2, seed=42)
train_dataset = train_temp['train']
test_dataset = train_temp['test']

In [ ]:
local_data = load_dataset(
    'csv',
    data_files={
        'train': f"{DATASET_INDO_CORPUS}train.tsv)",
        'test': f"{DATASET_INDO_CORPUS}test.tsv)"
    }
)

In [ ]:
# portion 90:10

train_temp = data.train_test_split(test_size=0.2, seed=42)
train_dataset = train_temp['train']
test_dataset = train_temp['test']


indo_speech = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})
print(indo_speech)

In [ ]:
processor = WhisperProcessor.from_pretrained(MODEL_NAME)

In [ ]:
indo_speech = indo_speech.cast_column('audio', Audio(sampling_rate=16000))

In [ ]:
def prep_dataset(batch):
  audio = batch['audio']
  batch['input_features'] = processor.feature_extractor(audio['array'], sampling_rate=audio['sampling_rate']).input_features[0]
  batch['labels'] = processor.tokenizer(batch['text']).input_ids
  return batch

In [ ]:
indo_speech = indo_speech.map(prep_dataset, remove_columns=indo_speech.column_names['train'], num_proc=2)

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
  def __init__(self, processor):
    self.processor = processor

  def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:

    input_feat = [{'input_features': feature['input_features']} for feature in features]
    batch = self.processor.feature_extractor.pad(input_feat, return_tensors='pt')

    label_feat = [{'input_ids': feature['labels']} for feature in features]
    label_batch = self.processor.tokenizer.pad(label_feat, return_tensors='pt')

    labels = label_batch['input_ids'].masked_fill(label_batch.attention_mask.ne(1), -100)

    if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
      labels = labels[:, 1:]

    batch['labels'] = labels

    return batch


In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
metric = evaluate.load('wer')

In [ ]:
bnb_conf = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False
)

whisp_model = WhisperForConditionalGeneration.from_pretrained(
    'Mufliramadhan/indo_speech',
    quantization_config=bnb_conf,
    device_map='auto',
    dtype=torch.float16
)

In [ ]:
whisp_model = prepare_model_for_kbit_training(whisp_model)

In [ ]:
lora_config = LoraConfig(r=32, lora_alpha=64, target_modules=['q_proj', 'v_proj'], lora_dropout=0.05, bias='none')
model = get_peft_model(whisp_model, lora_config)
model.print_trainable_parameters()

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_PATH,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    learning_rate=1e-3,
    warmup_steps=50,
    num_train_epochs=3,
    eval_strategy='steps',
    fp16=True,
    per_device_eval_batch_size=8,
    generation_max_length=128,
    logging_steps=100,
    remove_unused_columns=False,
    label_names=['labels']
)

In [ ]:
class SavePeftModelCallback(TrainerCallback):
  def on_save(
      self,
      args: TrainingArguments,
      state: TrainerState,
      control: TrainerControl,
      **kwargs
  ):
    checkpoint_folder = os.path.join(args.output_dir, f"{PREFIX_CHECKPOINT_DIR}-{state.global_step}")
    peft_model_path = os.path.join(checkpoint_folder, 'adapter_model')
    kwargs['model'].save_pretrained(peft_model_path)

    pytorch_model_path = os.path.join(checkpoint_folder, 'pytorch_model.bin')

    if os.path.exists(pytorch_model_path):
      os.remove(pytorch_model_path)
    return control



trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=indo_speech['train'],
    eval_dataset=indo_speech['test'],
    data_collator=data_collator,
    tokenizer=processor.feature_extractor,
    callbacks=[SavePeftModelCallback]
)


In [ ]:
trainer.train()

In [ ]:
peft_model_id = 'Mufliramadhan/indo_speech'
model.push_to_hub(peft_model_id)

# evaluate

In [ ]:
peft_model_id = 'Mufliramadhan/indo_speech'
peft_config = PeftConfig.from_pretrained(peft_model_id)
model = WhisperForConditionalGeneration.from_pretrained(peft_config.base_model_name_or_path, quantization_config=bnb_conf, device_map='auto', dtype=torch.float16)
model = PeftModel.from_pretrained(model, peft_model_id)
model.config.use_cache = True

In [ ]:
eval_dataloader = DataLoader(indo_speech['test'], batch_size=8, collate_fn=data_collator)
forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANG, task=TASK)
normalizer = BasicTextNormalizer()

pred = []
ref = []
norm_pred = []
norm_ref = []

model.eval()
for step, batch in enumerate(tqdm(eval_dataloader)):
  with torch.cuda.amp.autocast():
    with torch.no_grad():
      generated_tokens = (
          model.generate(
              input_features=batch['input_features'].to('cuda'),
              forced_decoder_ids=forced_decoder_ids,
              max_new_tokens=256
          ).cpu().numpy()
      )
      labels = batch['labels'].cpu().numpy()
      labels = np.where(labels != -100, labels, processor.tokenizer.pad_token_id)
      decod_preds = processor.tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
      decod_labels = processor.tokenizer.batch_decode(labels, skip_special_tokens=True)
      pred.extend(decod_preds)
      ref.extend(decod_labels)
      norm_pred.extend([normalizer(text).strip() for text in decod_preds])
      norm_ref.extend([normalizer(text).strip() for text in decod_labels])
    del generated_tokens, labels, batch
  gc.collect()

wer = 100 * metric.compute(predictions=pred, references=ref)
norm_wer = 100 * metric.compute(predictions=norm_pred, references=norm_ref)
eval_metrics = {'eval/wer': wer, "eval/normalized_wer"}
print(f'WER: {wer}')
print(f'Norm WER: {norm_wer}')
print(eval_metrics)

In [ ]:
eval_metrics = {'eval/wer': wer, "eval/normalized_wer": norm_wer}
print(eval_metrics)